# 11 报告模块 (report)

提供专业的模型报告生成功能，包括特征分箱统计、规则分析、SWAP分析、逾期预测、模型报告等。

**数据说明**: 基于 `hscredit_yyp.xlsx`，目标变量为 `MOB1 > 3`

In [ ]:
import os, sys
sys.path.append('../')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from hscredit import init_setting
from hscredit.report import (
    ExcelWriter, dataframe2excel, feature_bin_stats, feature_efficiency_analysis,
    ruleset_analysis, multi_label_rule_analysis,
    SwapAnalyzer, create_swap_dataset, SwapType,
    OverduePredictor, overdue_prediction_report,
    QuickModelReport, auto_model_report, compare_models,
    SingleFeatureRuleMiner, MultiFeatureRuleMiner, TreeRuleExtractor, MultiLabelRuleMiner,
    calculate_rule_metrics, RuleMetrics,
)
from hscredit.core.rules import Rule
from hscredit.core.binning import OptimalBinning
from hscredit.core.models import LogisticRegression
from sklearn.model_selection import train_test_split

init_setting()

df = pd.read_excel('hscredit_yyp.xlsx')
df['target'] = (df['MOB1'] > 3).astype(int)

numeric_features = ['中智小牛分C3', '珊瑚92', '极光欺诈分6v1', '青云24', '占信V3',
                   '轻花老客海纳子分V1', '天创小额网贷分', '衡枢鉴真分老客版']

df_model = df[numeric_features + ['target']].copy()
df_model = df_model.dropna()

print(f"样本数: {len(df_model):,}")
print(f"坏样本率: {df_model['target'].mean():.2%}")

## 1. Excel报告生成 (ExcelWriter + dataframe2excel)

In [ ]:
# 使用 ExcelWriter 创建多Sheet报告
with ExcelWriter().set_filename('demo_report.xlsx') as writer:
    # Sheet 1: 数据概览
    summary_df = pd.DataFrame({
        '指标': ['样本数', '特征数', '坏样本数', '坏样本率', '好样本数'],
        '值': [len(df_model), len(numeric_features), df_model['target'].sum(),
               f"{df_model['target'].mean():.2%}", len(df_model) - df_model['target'].sum()]
    })
    ws1 = writer.get_sheet_by_name('数据概览')
    writer.insert_df2sheet(ws1, summary_df, "B2")
    
    # Sheet 2: 特征统计
    stats_df = df_model[numeric_features].describe().reset_index()
    ws2 = writer.get_sheet_by_name('特征统计')
    writer.insert_df2sheet(ws2, stats_df, "B2")

print("Excel报告已生成: demo_report.xlsx")

In [ ]:
# 使用 dataframe2excel 直接导出 DataFrame
iv_results = []
for feat in numeric_features:
    feat_vals = df_model[feat].fillna(df_model[feat].median())
    from hscredit.core.metrics import iv
    iv_val = iv(df_model['target'], feat_vals, max_n_bins=10)
    iv_results.append({'特征': feat, 'IV': iv_val})

iv_df = pd.DataFrame(iv_results).sort_values('IV', ascending=False)
display(iv_df)

# 导出IV结果到Excel
dataframe2excel(iv_df, 'feature_iv.xlsx', sheet_name='特征IV值', index=False)
print("\nIV结果已导出: feature_iv.xlsx")

## 2. 特征分箱统计 (feature_bin_stats)

In [ ]:
# 单特征分析 - 使用 data, feature, target 参数
feature_name = '中智小牛分C3'

stats = feature_bin_stats(
    data=df_model,
    feature=feature_name,
    target='target',
    max_n_bins=10,
    method='mdlp'
)

print(f"=== {feature_name} 分箱统计 ===")
display(stats)

In [ ]:
# 多特征批量分析
features_to_analyze = ['中智小牛分C3', '珊瑚92', '极光欺诈分6v1']

multi_stats = feature_bin_stats(
    data=df_model,
    feature=features_to_analyze,
    target='target',
    method='quantile',
    max_n_bins=5
)

display(multi_stats)

In [ ]:
# 使用自定义分箱规则
stats_rules = feature_bin_stats(
    data=df_model,
    feature='中智小牛分C3',
    target='target',
    rules=[500, 600, 680, 750],
    desc='中智小牛分C3（自定义规则）'
)

display(stats_rules)

In [ ]:
# 逾期模式分析 - 使用 overdue + dpds 参数
stats_overdue = feature_bin_stats(
    data=df,
    feature='中智小牛分C3',
    overdue='MOB1',
    dpds=3,
    max_n_bins=5,
    del_grey=False
)

display(stats_overdue)

## 3. 特征效率分析 (feature_efficiency_analysis)

In [ ]:
# 对比手工分箱与自动分箱效果
# 首先需要数据中有目标列
df_tmp = df_model.copy()

result = feature_efficiency_analysis(
    data=df_tmp,
    feature='中智小牛分C3',
    manual_rules=[500, 600, 680, 750],
    target='target',
    auto_method='mdlp',
    max_n_bins=5,
)

print("手工分箱规则:", result['manual_rules'])
print("自动分箱规则:", result['auto_rules'])
plt.show()  # 显示组合图

## 4. 规则集分析 (ruleset_analysis)

In [ ]:
# 重命名列以便规则表达式使用
df_rules = df_model.rename(columns={
    '中智小牛分C3': 'score_c3',
    '珊瑚92': 'score_coral',
    '极光欺诈分6v1': 'score_fraud',
    '青云24': 'score_qingyun',
    '占信V3': 'score_zhanxin',
})

# 定义规则
rules = [
    Rule(expr="score_c3 < 550", name="低分规则", weight=100),
    Rule(expr="score_coral < 60", name="低珊瑚分", weight=80),
    Rule(expr="score_fraud > 0.7", name="高欺诈分", weight=90),
]

# 规则集综合分析
ruleset_report = ruleset_analysis(
    datasets=df_rules,
    rules=rules,
    target='target',
)

display(ruleset_report)

## 5. 多标签规则分析 (multi_label_rule_analysis)

In [ ]:
# 多标签规则分析 - 支持多个逾期标签
df_ml = df.copy()
df_ml['target_mob1'] = (df_ml['MOB1'] > 3).astype(int)
df_ml['target_mob3'] = (df_ml['MOB3'] > 3).astype(int)

features = ['中智小牛分C3', '珊瑚92', '极光欺诈分6v1']
labels = {'MOB1>3': 'target_mob1', 'MOB3>3': 'target_mob3'}

# 重命名列
df_ml = df_ml.rename(columns={
    '中智小牛分C3': 'score_c3',
    '珊瑚92': 'score_coral',
    '极光欺诈分6v1': 'score_fraud',
})
features_en = ['score_c3', 'score_coral', 'score_fraud']

# 注意：multi_label_rule_analysis 需要无缺失数据
df_ml_clean = df_ml.dropna(subset=features_en + list(labels.values()))
print(f"样本数（清洗后）: {len(df_ml_clean)}")

result_path = multi_label_rule_analysis(
    df=df_ml_clean,
    features=features_en,
    labels=labels,
    miner_params={'min_support': 0.02, 'min_lift': 1.3},
    output_path='multi_label_rules.xlsx'
)
print(f"多标签规则分析结果已保存: {result_path}")

## 6. SWAP分析 (SwapAnalyzer)

In [ ]:
# SWAP分析 - 评估新旧策略切换时的风险变化
df_swap = df_model.copy()
df_swap['score'] = df_swap['中智小牛分C3']

# 模拟新旧策略的通过/拒绝
np.random.seed(42)
df_swap['old_strategy'] = (df_swap['score'] > 600).astype(int)  # 旧策略: >600 通过
df_swap['new_strategy'] = (df_swap['score'] > 550).astype(int)  # 新策略: >550 通过

# 基础统计
print("=== SWAP 四象限统计 ===")
print(f"in-in (通过→通过): {((df_swap['old_strategy']==1) & (df_swap['new_strategy']==1)).sum()}")
print(f"in-out (通过→拒绝): {((df_swap['old_strategy']==1) & (df_swap['new_strategy']==0)).sum()}")
print(f"out-in (拒绝→通过): {((df_swap['old_strategy']==0) & (df_swap['new_strategy']==1)).sum()}")
print(f"out-out (拒绝→拒绝): {((df_swap['old_strategy']==0) & (df_swap['new_strategy']==0)).sum()}")

# out-in 样本的风险分析（这些是新策略引入的风险样本）
out_in_mask = (df_swap['old_strategy'] == 0) & (df_swap['new_strategy'] == 1)
out_in_bad_rate = df_swap.loc[out_in_mask, 'target'].mean() if out_in_mask.sum() > 0 else 0
overall_bad_rate = df_swap['target'].mean()
print(f"\nout-in 样本坏账率: {out_in_bad_rate:.2%}")
print(f"总体坏账率: {overall_bad_rate:.2%}")
print(f"out-in Lift: {out_in_bad_rate/overall_bad_rate:.2f}" if out_in_bad_rate > 0 else "")

## 7. 逾期预测 (OverduePredictor)

In [ ]:
# 逾期率预测器 - 基于分箱逾期率预估无标签样本
df_pred = df_model.copy()

predictor = OverduePredictor(
    feature='中智小牛分C3',
    target='target',
    method='quantile',
    max_n_bins=5,
)

# 拟合 - 从带标签数据学习分箱逾期率
predictor.fit(df_pred[['中智小牛分C3', 'target']], y=df_pred['target'])

# 转换 - 对全部样本预测逾期率
df_pred['pred_overdue_rate'] = predictor.transform(df_pred[['中智小牛分C3']].copy())

print("=== 逾期率预测结果 ===")
display(df_pred[['中智小牛分C3', 'target', 'pred_overdue_rate']].head(10))

print(f"\n各分箱逾期率映射:")
for label, rate in predictor.bin_rates_.items():
    print(f"  {label}: {rate:.4f}")

## 8. 规则挖掘 (SingleFeatureRuleMiner, MultiFeatureRuleMiner, TreeRuleExtractor)

In [ ]:
# 单特征规则挖掘
df_mining = df_model.rename(columns={
    '中智小牛分C3': 'score_c3',
    '珊瑚92': 'score_coral',
})

# 准备无缺失数据
df_mining = df_mining.dropna(subset=['score_c3', 'score_coral', 'target'])

# 单特征规则挖掘
single_miner = SingleFeatureRuleMiner(
    feature='score_c3',
    target='target',
    max_bins=5,
    min_support=0.05,
    min_lift=1.2
)
single_miner.fit(df_mining)

rules_single = single_miner.get_rules()
print(f"=== 单特征(score_c3)规则挖掘: 共 {len(rules_single)} 条规则 ===")
if len(rules_single) > 0:
    display(pd.DataFrame(rules_single).head(10))

In [ ]:
# 多特征规则挖掘
multi_miner = MultiFeatureRuleMiner(
    features=['score_c3', 'score_coral'],
    target='target',
    max_depth=2,
    min_support=0.03,
    min_lift=1.5
)
multi_miner.fit(df_mining)

rules_multi = multi_miner.get_rules()
print(f"=== 多特征规则挖掘: 共 {len(rules_multi)} 条规则 ===")
if len(rules_multi) > 0:
    display(pd.DataFrame(rules_multi).head(10))

In [ ]:
# 决策树规则提取
tree_extractor = TreeRuleExtractor(
    max_depth=3,
    min_samples_leaf=50,
    max_leaf_nodes=8
)

X_tree = df_mining[['score_c3', 'score_coral']].fillna(df_mining[['score_c3', 'score_coral']].median())
y_tree = df_mining['target']

tree_extractor.fit(X_tree, y_tree)

# 提取规则
tree_rules = tree_extractor.extract_rules()
print(f"=== 决策树提取规则: 共 {len(tree_rules)} 条 ===")
for rule in tree_rules:
    print(f"  {rule['rule']} -> lift: {rule['lift']:.2f}, support: {rule['support']:.2%}")

## 9. 规则评估指标 (calculate_rule_metrics)

In [ ]:
# 计算规则的详细评估指标
test_rules = [
    Rule(expr="score_c3 < 550", name="低分规则"),
    Rule(expr="score_coral > 80", name="高风险规则"),
]

metrics_results = []
for rule in test_rules:
    mask = rule.predict(df_mining)
    metrics = calculate_rule_metrics(df_mining['target'], mask)
    metrics_results.append({
        '规则': rule.name,
        '覆盖率': metrics.support,
        '坏样本率': metrics.bad_rate,
        'Lift': metrics.lift,
        '精确率': metrics.precision,
        '召回率': metrics.recall,
        'F1': metrics.f1,
    })

display(pd.DataFrame(metrics_results))

## 10. 快速模型报告 (QuickModelReport / auto_model_report)

In [ ]:
# 快速模型报告
df_report = df_model.rename(columns={
    '中智小牛分C3': 'score_c3',
    '珊瑚92': 'score_coral',
})

selected_features = ['score_c3', 'score_coral', '极光欺诈分6v1']
df_report = df_report.dropna(subset=selected_features + ['target'])

X = df_report[selected_features]
y = df_report['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 训练简单模型
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]

print("=== QuickModelReport 快速模型评估 ===")
from hscredit.core.metrics import ks, auc
print(f"KS: {ks(y_test, y_prob):.4f}")
print(f"AUC: {auc(y_test, y_prob):.4f}")

In [ ]:
# auto_model_report - 自动生成完整模型报告（可输出到Excel）
report = auto_model_report(
    model=model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    feature_names=selected_features,
    output_dir='model_report',
)
print("模型报告已生成到 model_report/ 目录")

In [ ]:
# compare_models - 对比多个模型
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=50, random_state=42, max_depth=3)
rf.fit(X_train, y_train)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

comparison = compare_models(
    models={'LR': model, 'RF': rf},
    X_test=X_test,
    y_test=y_test,
)

print("=== 模型对比结果 ===")
display(comparison)

## 11. 分箱方法对比 (benchmark_binning_methods)

In [ ]:
# 对比多种分箱方法效果
from hscredit.report.feature_analyzer import benchmark_binning_methods

df_bench = df_model.copy()

benchmark = benchmark_binning_methods(
    data=df_bench,
    feature='中智小牛分C3',
    overdue_col='MOB1',
    dpds=[3, 0],
    max_n_bins=5,
)

display(benchmark[['method', 'dpd', 'n_bins', 'head_lift', 'tail_lift', 'edge_gap', 'monotonic']].head(10))